# Sistema di Visione 3D

# 01 Distorsione & Crop

In [22]:
import os
import pickle
import numpy as np
import cv2 as cv
import pickle
from os import listdir, path
from matplotlib import pyplot as plt

# Directory e costanti
IMGDIR      = 'images'
TARGET      = os.path.join(IMGDIR,'target')
LAMA        = os.path.join(IMGDIR,'lama')
OGGETTO     = os.path.join(IMGDIR,'oggetto')
CROP        = 'cropped'
OUTDIR      = 'outputs'  # Directory per salvare i risultati
MAX_ROWS    = 14         # Numero massimo di righe della griglia
MAX_COLS    = 21         # Numero massimo di colonne della griglia

# Creazione delle cartelle di output se non esistenti
if not path.exists(OUTDIR):
    os.makedirs(OUTDIR)

if not path.exists(os.path.join(TARGET,CROP)):
    os.makedirs(os.path.join(TARGET,CROP))

if not path.exists(os.path.join(LAMA,CROP)):
    os.makedirs(os.path.join(LAMA,CROP))
    
if not path.exists(os.path.join(OGGETTO,CROP)):
    os.makedirs(os.path.join(OGGETTO,CROP))

In [23]:
def vardump(var, fname):
    """
    Serializza un oggetto Python e lo salva in un file binario.

    Parametri:
      var   : Oggetto Python da salvare su disco.
      fname : Percorso e nome del file binario in cui salvare 'var'.

    Note:
      - Utilizza la libreria 'pickle' per la serializzazione.
      - La modalità 'wb' indica scrittura in binario.
    """
    with open(fname, 'wb') as fw:
        pickle.dump(var, fw)

In [ ]:
main_image_path = None

# Ricerca di un file JPG contenente la stringa 'main' all'interno della cartella TARGET
for f in sorted(os.listdir(TARGET)):
    if f.lower().endswith('.jpg') and 'main' in f:
        main_image_path = os.path.join(TARGET, f)
        print(f"Immagine principale trovata: {f}")
        break

# Se non viene trovata alcuna immagine valida
assert main_image_path is not None, "ERRORE: Immagine 'main' non presente nella cartella!"

# Caricamento dell'immagine principale
img = cv.imread(main_image_path, cv.IMREAD_COLOR)
print("Dimensioni immagine originale (H, W, C):", img.shape)

# Calcolo del rapporto altezza/larghezza e ridimensionamento
h, w, _ = img.shape
ratio = h / w
w_new = w  # Mantiene la larghezza originale (o modificarla se necessario)
dim = (w_new, int(w_new * ratio))
imgBGR = cv.resize(img, dim)
print("Dimensioni immagine ridimensionata (H, W, C):", imgBGR.shape)

# Conversioni di colore: BGR->RGB e BGR->Gray
imgRGB = cv.cvtColor(imgBGR, cv.COLOR_BGR2RGB)
imgGray = cv.cvtColor(imgBGR, cv.COLOR_BGR2GRAY)

# Calcolo della distribuzione percentuale delle intensità in scala di grigio
H, W = imgGray.shape
hist_vals, _ = np.histogram(imgGray, bins=range(257))
pdf = (hist_vals / float(H * W)) * 100

In [ ]:
# Creazione di una figura con layout 1x3, dimensioni di 15x5 pollici
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Primo subplot: visualizzazione a colori (RGB)
axs[0].imshow(imgRGB)
axs[0].set_title("Immagine a Colori")
axs[0].set_xticks([])
axs[0].set_yticks([])

# Secondo subplot: istogramma delle intensità (pdf)
axs[1].plot(pdf)
axs[1].set_title("Istogramma dell'Immagine")
axs[1].set_xlim([0, 256])
axs[1].set_xlabel("Intensità di Colore")
axs[1].set_ylabel("Percentuale di Pixel")

# Terzo subplot: visualizzazione in scala di grigi
axs[2].imshow(imgGray, cmap='gray')
axs[2].set_title("Immagine in Scala di Grigi")
axs[2].set_xticks([])
axs[2].set_yticks([])

plt.tight_layout()
plt.show()

## Ricerca Pattern

In [ ]:
def find_first_valid_circles_grid(img_gray, max_rows = MAX_ROWS, max_cols = MAX_COLS):
    """
    Cerca il primo pattern di cerchi valido in un'immagine in scala di grigi,
    partendo dalle dimensioni di griglia più grandi e decrescendo fino a
    trovare un match.

    Parametri:
      img_gray (np.array): immagine in scala di grigi.
      max_rows (int)     : numero massimo di righe (default=20).
      max_cols (int)     : numero massimo di colonne (default=30).

    Ritorna:
      best_pattern (tuple): pattern (righe, colonne) del primo pattern trovato.
      centers      (np.array): coordinate dei centri trovati, shape (n, 1, 2).
      found_num    (int)      : numero di cerchi trovati per il pattern.

    Note:
      - Utilizza cv.findCirclesGrid con la modalità CALIB_CB_SYMMETRIC_GRID.
      - Restituisce None se non individua alcun pattern.
    """
    # Inizia dai pattern più grandi verso i più piccoli
    for c in range(max_cols, 9, -1):
        for r in range(max_rows, 9, -1):
            pattern = (c, r)
            ret, centers = cv.findCirclesGrid(
                img_gray,
                pattern,
                flags=cv.CALIB_CB_SYMMETRIC_GRID
            )
            if ret and centers is not None:
                return pattern, centers, len(centers)

    # Se non viene trovato alcun pattern
    return None, None, 0


def draw_circles_on_image(img_rgb, centers):
    """
    Disegna e visualizza i centri di un pattern di cerchi su un'immagine RGB.

    Parametri:
      img_rgb (np.array) : immagine a colori (RGB).
      centers (np.array) : array di centri individuati (n, 1, 2).

    Note:
      - I centri vengono disegnati come cerchi pieni di colore fucsia.
      - L'immagine finale viene visualizzata con matplotlib.
    """
    cdst = img_rgb.copy()
    for c_pt in centers:
        cx, cy = c_pt[0].astype(int)
        cv.circle(cdst, (cx, cy), 12, (255, 0, 255), -1)

    fig, ax = plt.subplots()
    ax.imshow(cdst)
    ax.axis('off')
    plt.show()


# Esempio di utilizzo
pattern, centers, found_num = find_first_valid_circles_grid(imgGray)

if pattern is None:
    print("Nessun pattern trovato!")
else:
    # Calcolo del numero totale di cerchi previsti
    total_cerchi = pattern[0] * pattern[1]
    print(f"Primo pattern valido trovato: {pattern[1]} righe x {pattern[0]} colonne")
    print(f"Cerchi effettivamente trovati: {found_num} / {total_cerchi}")
    draw_circles_on_image(imgRGB, centers)

In [ ]:
# Creazione dei punti 3D (x, y, z=0) sul piano del pattern
objp = np.zeros((total_cerchi, 3), dtype=np.float32)
# np.mgrid[0:r, 0:c] crea due matrici 2D; T.reshape(-1, 2) le converte in un array Nx2
objp[:, :2] = np.mgrid[0:pattern[0], 0:pattern[1]].T.reshape(-1, 2)

# Liste dei punti 3D (reali) e 2D (immagine)
objpoints = [objp]
imgpoints = [centers]

# Calibrazione della telecamera con i punti 3D/2D
ret, cameraMatrix, distCoeffs, rvecs, tvecs = cv.calibrateCamera(
    objpoints,
    imgpoints,
    imgGray.shape,
    None,
    None
)

print("Esito Calibrazione:", ret)
print("Matrice Intrinseca (cameraMatrix):")
display(cameraMatrix)
print("Coefficienti di Distorsione (distCoeffs):")
display(distCoeffs)

## Eliminazione Distorsione

In [ ]:
# Calcolo della nuova matrice intrinseca e della ROI ottimizzata
newcameramtx, roi = cv.getOptimalNewCameraMatrix(
    cameraMatrix,
    distCoeffs,
    (w, h),
    1, 
    (w, h)
)
print("Matrice Intrinseca Ottimizzata:\n", newcameramtx)

# Copia dell'immagine RGB iniziale e conversione a scala di grigi
img2 = imgRGB.copy()
gray2 = cv.cvtColor(img2, cv.COLOR_RGB2GRAY)

# Rimozione della distorsione ottica tramite i parametri di calibrazione
undistorted = cv.undistort(img2, cameraMatrix, distCoeffs, None, newcameramtx)

# Ritaglio dell'immagine in base alla ROI restituita
x, y, w, h = roi
cropped = undistorted[y:y+h, x:x+w]

# Visualizzazione delle tre fasi: Originale, Undistorted, Cropped
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
fig = plt.figure()

axs[0].imshow(gray2, cmap='gray')
axs[0].axis('off')
axs[0].set_title("Original (Gray)")

axs[1].imshow(undistorted)
axs[1].axis('off')
axs[1].set_title("Undistorted")

axs[2].imshow(cropped)
axs[2].axis('off')
axs[2].set_title("Cropped")

plt.tight_layout()
plt.show()

# Calcolo dell'errore medio di calibrazione
mean_error = 0
for i in range(len(objpoints)):
    # Proiezione dei punti 3D sui piani immagine
    imgpoints2, _ = cv.projectPoints(
        objpoints[i],
        rvecs[i],
        tvecs[i],
        cameraMatrix,
        distCoeffs
    )
    # Calcolo dell'errore medio euclideo tra i punti trovati e quelli proiettati
    error = cv.norm(imgpoints[i], imgpoints2, cv.NORM_L2) / len(imgpoints2)
    mean_error += error

print("Errore medio di calibrazione:", mean_error / len(objpoints))

In [ ]:
# Copia dell'immagine 'undistorted' su cui disegnare la ROI
img_with_lines = undistorted.copy()
color = (0, 255, 0)  # Colore verde per le linee
thickness = 2        # Spessore delle linee

# Disegno delle linee verticali lungo i bordi della ROI
cv.line(img_with_lines, (x, y), (x, y + h), color, thickness)
cv.line(img_with_lines, (x + w, y), (x + w, y + h), color, thickness)

# Disegno delle linee orizzontali lungo i bordi della ROI
cv.line(img_with_lines, (x, y), (x + w, y), color, thickness)
cv.line(img_with_lines, (x, y + h), (x + w, y + h), color, thickness)

# Visualizzazione dell'immagine con le linee ROI
plt.imshow(cv.cvtColor(img_with_lines, cv.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Undistorted Image with ROI")
plt.show()

# Creazione di una versione 'cropped' (ritagliata) con le linee visibili
cropped_with_lines = img_with_lines[y:y+h, x:x+w]

# Visualizzazione dell'area ritagliata con linee incluse
plt.imshow(cv.cvtColor(cropped_with_lines, cv.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Cropped ROI with Lines")
plt.show()

## Ricerca Crop Ottimale

In [ ]:
# Ricerca del pattern di cerchi più grande nell'immagine 'cropped'
pattern, centers, found_num = find_first_valid_circles_grid(cropped)

if pattern is None:
    print("Nessun pattern valido trovato nella regione 'cropped'.")
else:
    # Informazioni sul pattern identificato
    total_cerchi = pattern[0] * pattern[1]
    print(f"Pattern trovato: {pattern[1]} righe x {pattern[0]} colonne")
    print(f"Cerchi rilevati: {found_num} / {total_cerchi}")

    # Disegno dei centri dei cerchi per verifica visiva
    draw_circles_on_image(cropped, centers)

    # Conversione di 'centers' in coordinate 2D
    imgpoints = centers.reshape(-1, 2)

    # Calcolo del bounding box dei centri
    x_coords = imgpoints[:, 0]
    y_coords = imgpoints[:, 1]
    x_min, x_max = int(np.min(x_coords)), int(np.max(x_coords))
    y_min, y_max = int(np.min(y_coords)), int(np.max(y_coords))

    # Applicazione di un padding facoltativo
    padding = 30
    x_min = max(0, x_min - padding)
    x_max = min(cropped.shape[1], x_max + padding)
    y_min = max(0, y_min - padding)
    y_max = min(cropped.shape[0], y_max + padding)

    # Ritaglio dell'area circostante il pattern
    cropped_img = cropped[y_min:y_max, x_min:x_max]

    # Informazioni di debug sulle dimensioni
    print("Dim. originale del 'cropped':", cropped.shape)
    print("Dim. immagine ulteriormente ritagliata:", cropped_img.shape)

    # Coordinata assoluta della nuova ROI (riferita a 'undistorted')
    roi_x, roi_y = x + x_min, y + y_min
    roi_w = x_max - x_min
    roi_h = y_max - y_min
    roi = (roi_x, roi_y, roi_w, roi_h)
    print("Nuova ROI (assoluta):", roi)

    # Visualizzazione del risultato
    plt.figure(figsize=(8, 6))
    plt.imshow(cv.cvtColor(cropped_img, cv.COLOR_BGR2RGB))
    plt.title("Immagine Croppata con Pattern")
    plt.axis('off')
    plt.show()

## Undistorted e Crop per Ogni Immagine

In [31]:
def crop_and_undistort_image(directory, cameraMatrix = cameraMatrix, distCoeffs = distCoeffs, newcameramtx = newcameramtx, roi = roi):
    """
    Esegue la correzione della distorsione ottica e il ritaglio (ROI) su ogni
    immagine JPG presente nella cartella specificata, quindi salva il risultato
    in una sottocartella 'cropped' con un suffisso '_CROPPED'.

    Parametri:
      directory     (str)       : Percorso della cartella contenente le immagini da elaborare.
      cameraMatrix  (np.array)  : Matrice intrinseca della telecamera (3x3).
      distCoeffs    (np.array)  : Coefficienti di distorsione (1D).
      newcameramtx  (np.array)  : Matrice intrinseca ottimizzata, ottenuta con getOptimalNewCameraMatrix.
      roi           (tuple)     : Coordinate e dimensioni del ritaglio (x, y, w, h).

    Note:
      - Vengono elaborate solo le immagini con estensione .jpg (case-insensitive).
      - I file ricampionati vengono salvati in una sottocartella 'cropped',
        con il nome originale più il suffisso '_CROPPED'.
    """
    # Verifica e crea la cartella 'cropped' all'interno di 'directory' se non esiste
    cropped_dir = os.path.join(directory, CROP)
    if not os.path.exists(cropped_dir):
        os.makedirs(cropped_dir)

    for f in sorted(os.listdir(directory)):
        if f.lower().endswith('.jpg'):
            img_path = os.path.join(directory, f)
            img = cv.imread(img_path, cv.IMREAD_COLOR)
            if img is None:
                print(f"Impossibile leggere l'immagine: {f}")
                continue

            print(f"Elaborazione immagine: {f}")

            # Correzione della distorsione
            undistorted = cv.undistort(img, cameraMatrix, distCoeffs, None, newcameramtx)

            # Ritaglio della porzione ROI
            x, y, w, h = roi
            cropped = undistorted[y:y+h, x:x+w]

            # Creazione del nuovo nome file con suffisso
            name, ext = os.path.splitext(f)
            new_filename = f"{name}_CROPPED{ext}"
            save_path = os.path.join(cropped_dir, new_filename)

            # Salvataggio dell'immagine ritagliata
            cv.imwrite(save_path, cropped)
            print(f"Immagine salvata come: {new_filename}")

In [ ]:
crop_and_undistort_image(TARGET)
crop_and_undistort_image(LAMA)
crop_and_undistort_image(OGGETTO)

A causa del setup di misura non perfettamente stabile è stato scelto di utilizzare solo una parte del dataset di informazioni raccolte, in particolare le immagini con z <= z main.
In questo modo si riesce ad ottenere misurazioni più accurate.


In [ ]:
def remove_images_with_target_z(target_dir, remove = True):
    """
    Rimuove tutte le immagini con z maggiore/minore di z main dalla directory specificata.

    Parametri:
      target_dir (str): Percorso della cartella contenente le immagini da elaborare.
      remove (Boolean): True rimuove immagini con z > z main
                        False rimuove immagini con z < z main
    """
    for f in sorted(os.listdir(TARGET)):
        if f.lower().endswith('.jpg') and 'main' in f:
            z_main = int(f[:2])
            break

    # Rimuovi le immagini in base al valore di z
    for f in sorted(os.listdir(target_dir)):
        if f.lower().endswith('.jpg') and 'main' not in f:
            try:
                z_value = int(f[:2])
                if (remove and z_value > z_main) or (not remove and z_value < z_main):
                    os.remove(os.path.join(target_dir, f))
                    print(f"Immagine rimossa: {f}")
            except (IndexError, ValueError):
                print(f"Impossibile estrarre il valore di z dall'immagine: {f}")

remove = True
remove_images_with_target_z(os.path.join(TARGET, CROP))
remove_images_with_target_z(os.path.join(LAMA, CROP))

In [34]:
vardump(cameraMatrix, path.join(OUTDIR, 'cameraMatrix1.dump'))
# Salva la matrice intrinseca della telecamera in un file binario chiamato "cameraMatrix1.dump".
# La matrice `cameraMatrix` contiene informazioni cruciali come lunghezze focali e il punto principale.

vardump(distCoeffs, path.join(OUTDIR, 'distCoeffs1.dump'))
# Salva i coefficienti di distorsione della telecamera in un file binario chiamato "distCoeffs1.dump".
# I coefficienti di distorsione sono utilizzati per correggere le distorsioni ottiche nelle immagini.

vardump(newcameramtx, path.join(OUTDIR, 'newcameramtx1.dump'))
# Salva la nuova matrice intrinseca ottimizzata in un file binario chiamato "newcameramtx1.dump".
# Questa matrice tiene conto della distorsione eliminata per migliorare l'area utile dell'immagine.

vardump(roi, path.join(OUTDIR, 'roi1.dump')) 
# Salva la regione utile della nuova immagine senza distorsioni in un file binario chiamato "roi1.dump".
# `roi` (region of interest) definisce i confini dell'immagine corretta.